# DSA4153 Week 10: Four-Format Data Recovery

This notebook imports the project modules rather than reimplementing the processing logic. The input data are historical Gapminder observations for 1952–2007. Run from the repository root after installing `requirements.txt`.

**Reconciliation target:** per input, 1,712 raw records = 6 rejected + 2 exact duplicate copies removed + 1,704 canonical rows. Across four formats, 24 quarantined records and 8 duplicate copies are expected.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np

# Locate the repository root
ROOT = Path.cwd().resolve()

if not (ROOT / "src").is_dir():
    ROOT = (ROOT / "..").resolve()

# Add repository root to Python's import path
sys.path.insert(0, str(ROOT))

# Import project modules
from src.recovery import recover_one, assert_agreement
from src.analysis import create_analyses

manifest = json.loads((ROOT / "data/raw/manifest.json").read_text(encoding="utf-8"))
config = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))

print("Project root:", ROOT)
print("Canonical columns:", manifest["canonical_order"])
print("Allowed years:", manifest["allowed_years"])
print("Tolerance:", config["comparison"])

Project root: /Users/banjo/Downloads/DSA4153-Week10-Group-Task
Canonical columns: ['country', 'continent', 'year', 'lifeExp', 'pop', 'gdpPercap']
Allowed years: [1952, 1957, 1962, 1967, 1972, 1977, 1982, 1987, 1992, 1997, 2002, 2007]
Tolerance: {'absolute_tolerance': 1e-08, 'relative_tolerance': 0}


In [2]:
bundles = [
    recover_one(ROOT / "data/raw" / filename, manifest)
    for filename in config["inputs"]
]
canonical = assert_agreement(bundles, atol=config["comparison"]["absolute_tolerance"])
pd.DataFrame([
    {"source": b.source_file, "raw": b.raw_count, "quarantined": len(b.quarantine),
     "duplicate_copies_removed": len(b.duplicates), "canonical_rows": len(b.canonical)}
    for b in bundles
])

,source,raw,quarantined,duplicate_copies_removed,canonical_rows
0,country_records.csv,1712,6,2,1704
1,country_records.xlsx,1712,6,2,1704
2,country_records.json,1712,6,2,1704
3,country_records.parquet,1712,6,2,1704


## Agreement checks

The recovery function sorts each version by `(country, year)` and the agreement function compares keys, text, integer values, and floating-point values with absolute tolerance `1e-8` and relative tolerance `0`. A conflicting key is an error; the program must not keep the first version silently.

In [3]:
analyses = create_analyses(canonical, int(config.get("reporting_year", 2007)))
print("Canonical shape:", canonical.shape)
print("Asian selected-year rows:", len(analyses["asia_selected_year"]))
print("Philippine history rows:", len(analyses["country_history"]))
display(analyses["asia_selected_year"].head())
display(analyses["country_history"])
display(analyses["population_index"])

Canonical shape: (1704, 6)
Asian selected-year rows: 33
Philippine history rows: 12


,country,continent,year,lifeExp,pop,gdpPercap,population_millions,lifeExp_rank
0,Japan,Asia,2007,82.603,127467972,31656.06806,127.467972,1
1,"Hong Kong, China",Asia,2007,82.208,6980412,39724.97867,6.980412,2
2,Israel,Asia,2007,80.745,6426679,25523.27710,6.426679,3
3,Singapore,Asia,2007,79.972,4553009,47143.17964,4.553009,4
4,"Korea, Rep.",Asia,2007,78.623,49044790,23348.13973,49.044790,5


,country,continent,year,lifeExp,pop,gdpPercap,population_millions,lifeExp_change_from_first
0,Philippines,Asia,1952,47.752,22438691,1272.880995,22.438691,0.000
1,Philippines,Asia,1957,51.334,26072194,1547.944844,26.072194,3.582
2,Philippines,Asia,1962,54.757,30325264,1649.552153,30.325264,7.005
3,Philippines,Asia,1967,56.393,35356600,1814.127430,35.356600,8.641
4,Philippines,Asia,1972,58.065,40850141,1989.374070,40.850141,10.313
5,Philippines,Asia,1977,60.060,46850962,2373.204287,46.850962,12.308
6,Philippines,Asia,1982,62.082,53456774,2603.273765,53.456774,14.330
7,Philippines,Asia,1987,64.151,60017788,2189.634995,60.017788,16.399
8,Philippines,Asia,1992,66.458,67185766,2279.324017,67.185766,18.706
9,Philippines,Asia,1997,68.564,75012988,2536.534925,75.012988,20.812


,country,1982,1987,1992,1997,2002,2007
0,Philippines,100.0,112.273494,125.682418,140.324570,155.256447,170.375577
1,Indonesia,100.0,110.390432,120.524576,129.955720,137.639149,145.782331
2,Thailand,100.0,108.362522,116.056504,123.326192,128.630762,133.262203


## NumPy population index

The matrix uses Philippines, Indonesia, and Thailand for 1982, 1987, 1992, 1997, 2002, and 2007. Each row is divided by its 1982 baseline and multiplied by 100. The processing module demonstrates broadcasting, a Boolean selection and aggregation, and a separately written loop calculation to check the result.

In [4]:
years = [1982, 1987, 1992, 1997, 2002, 2007]
print("Population index table shape (including country label):", analyses["population_index"].shape)
print("Broadcast matrix shape:", analyses["population_index"].shape[0], "x", len(years))
print("2007 Asia population:", int(canonical.loc[(canonical.continent == "Asia") & (canonical.year == 2007), "pop"].sum()))
print("Philippines 2007 population:", int(canonical.loc[(canonical.country == "Philippines") & (canonical.year == 2007), "pop"].iloc[0]))

Population index table shape (including country label): (3, 7)
Broadcast matrix shape: 3 x 6
2007 Asia population: 3811953827
Philippines 2007 population: 91077287


## Interpretation and exports

- The four source files are alternative representations of the same observations, not four independent datasets to concatenate.
- Quarantined records are invalid classroom additions; they are preserved with their source filename, one-based data-record position, reason, and original field values.
- Exact duplicates are logged, while conflicting `(country, year)` keys stop processing.
- The command-line program writes the canonical exports and analysis reports to a staging folder, reloads all four canonical formats, and promotes the results only when the validation report passes.
- Change `reporting_year` in `config.json` to `1997` and rerun to regenerate the Asian selected-year analysis without editing processing functions.

In [5]:
# Re-run the complete staged program from a terminal:
# python main.py --config config.json
# Export this notebook to HTML after running all cells:
# jupyter nbconvert --to html notebooks/report.ipynb --output report.html